# Module 3a — Generate evaluation questions from your library

**Time:** pilot ~10 min; calibration set takes longer · **Output:** pilot files or separate calibration files

1. Read and chunk the same documents you uploaded to MyGPT
2. Generate answerable question + reference-answer pairs
3. Add unanswerable/off-topic questions
4. **Review every question**, then run them through MyGPT
5. After the 03b Cohen's κ pilot, set `CALIBRATION_MODE = True` to create a separate set of 50+ questions for module 4

In [1]:
import sys
from pathlib import Path
import re
import random

# Make the repo root importable whether Jupyter was started from the root or this folder
ROOT = Path.cwd() if (Path.cwd() / "common").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from common.mygpt_client import MyGPTClient, outputs_dir, load_fallback_pubmedqa  # noqa: E402
OUT = outputs_dir()

In [2]:
DOCS_DIR = ROOT / "data" / "BioASQ" / "pdfs_70"
CALIBRATION_MODE = True               # set True after checking judge agreement in 03b
TARGET_QUESTIONS = 50                     # total, including unanswerable questions
N_UNANSWERABLE = 8                        # calibration set: choose 5–10 plausible out-of-scope questions
if CALIBRATION_MODE and (TARGET_QUESTIONS < 50 or not 5 <= N_UNANSWERABLE <= 10):
    raise ValueError("Calibration mode requires 50+ total questions and 5–10 unanswerable questions")
N_CHUNKS = TARGET_QUESTIONS - N_UNANSWERABLE if CALIBRATION_MODE else 20
QUESTION_FILE = OUT / ("03_calibration_questions.csv" if CALIBRATION_MODE else "03_questions.csv")
ANSWER_FILE = OUT / ("03_calibration_answers.csv" if CALIBRATION_MODE else "03_answers.csv")
CHUNK_SIZE = 1500                          # characters per chunk used for question generation
SEED = 42

client = MyGPTClient.from_env()
print("Generator LLM:", client.llm, "via", client.ollama_url)

Generator LLM: llama3:latest via http://host.docker.internal:11434


## 1. Read and chunk documents

In [3]:
def read_text(path: Path) -> str:
    suffix = path.suffix.lower()
    if suffix in {".txt", ".md"}:
        return path.read_text(encoding="utf-8", errors="ignore")
    if suffix == ".pdf":
        from pypdf import PdfReader
        return "\n".join(page.extract_text() or "" for page in PdfReader(path).pages)
    if suffix == ".docx":
        import docx
        return "\n".join(p.text for p in docx.Document(path).paragraphs)
    return ""

chunks = []
for path in sorted(DOCS_DIR.glob("*")):
    text = re.sub(r"\s+", " ", read_text(path)).strip()
    for start in range(0, len(text), CHUNK_SIZE):
        piece = text[start:start + CHUNK_SIZE]
        if len(piece) > 400:                      # skip tiny tail chunks
            chunks.append({"source_doc": path.name, "chunk": piece})

print(len(chunks), "chunks from", len({c["source_doc"] for c in chunks}), "documents")
if len(chunks) < N_CHUNKS:
    raise ValueError(f"Need {N_CHUNKS} source chunks, but found {len(chunks)}; add documents or reduce TARGET_QUESTIONS")
random.seed(SEED)
sample = random.sample(chunks, N_CHUNKS)

755 chunks from 70 documents


## 2. Generate question + reference answer per chunk
The prompt asks for **specific, self-contained** questions: a reader shouldn't need to see the chunk to understand the question.

In [4]:
import json

QGEN_PROMPT = """You are building a benchmark for a question-answering system over a document library.
Read the passage and write ONE question that:
- can be answered from this passage alone,
- is specific and self-contained (do not say "the passage", "this document", "the author"),
- a researcher in this field would realistically ask.
Then write a concise reference answer (1-2 sentences) using only the passage.

Return JSON: {{"question": "...", "answer": "..."}}

PASSAGE:
{chunk}
"""

rows = []
for i, item in enumerate(sample, 1):
    try:
        raw = client.ollama(QGEN_PROMPT.format(chunk=item["chunk"]), temperature=0.2, json_mode=True)
        qa = json.loads(raw)
        rows.append({"question": qa["question"].strip(), "reference_answer": qa["answer"].strip(),
                     "answerable": 1, "source_doc": item["source_doc"], "source_chunk": item["chunk"]})
        print(f"[{i}] {qa['question']}")
    except Exception as exc:
        print(f"[{i}] skipped: {exc}")

if len(rows) != N_CHUNKS:
    raise RuntimeError(f"Generated {len(rows)} of {N_CHUNKS} answerable questions; rerun generation or check the LLM before proceeding")

[1] What types of gastrointestinal cancers are associated with metabolic dysfunction-associated steatotic liver disease (MASLD)?
[2] What is the mean age of patients in the study?
[3] What are the most common symptoms of postinfectious irritable bowel syndrome (PI-IBS)?
[4] What is the result of comparing the size of WJSC-MT and FIB-MT over 28 days?
[5] What is the effect of PTEN loss on glioma cells' ability to escape the immune system?
[6] What percentage of glioblastoma patients showed EGFR gene amplification?
[7] What is the minimum viral load required for vector mosquitoes to be infected with dengue virus?
[8] What is the outcome of the precancerous cell's uncontrolled proliferation in the third stage?
[9] What drives the recurrence of glioblastoma multiforme (GBM)?
[10] What percentage of participants had a diagnosis of Generalized Anxiety Disorder?
[11] What percentage of children born outside the hospital of residence had the CMVc screening test ordered in a timely manner?
[12]

## 3. Add unanswerable questions
For the pilot, use 3 questions. For calibration mode, include 5–10 plausible but out-of-scope questions (8 by default); replace these examples with questions tailored to your library. They should be questions your documents **do not** answer, so they test whether HI catches hallucinations.

In [5]:
import pandas as pd

PILOT_UNANSWERABLE = [
    "What is the boiling point of tungsten?",
    "Who won the 2018 FIFA World Cup?",
    "What dose of imatinib is recommended for adults with chronic myeloid leukemia?",
]
CALIBRATION_UNANSWERABLE = [
    "What is the boiling point of tungsten?",
    "Who won the 2018 FIFA World Cup?",
    "What dose of imatinib is recommended for adults with chronic myeloid leukemia?",
    "What is the capital of Iceland?",
    "Who wrote Pride and Prejudice?",
    "How far is the Moon from Earth?",
    "How many rings does Saturn have?",
    "How do you make a sourdough starter?",
]
UNANSWERABLE = CALIBRATION_UNANSWERABLE if CALIBRATION_MODE else PILOT_UNANSWERABLE
if CALIBRATION_MODE and len(UNANSWERABLE) != N_UNANSWERABLE:
    raise ValueError(f"Expected {N_UNANSWERABLE} unanswerable examples; edit CALIBRATION_UNANSWERABLE")
NOT_IN_LIBRARY = "The library does not contain this information."
rows += [{"question": q, "reference_answer": NOT_IN_LIBRARY, "answerable": 0,
          "source_doc": "", "source_chunk": ""} for q in UNANSWERABLE]

questions = pd.DataFrame(rows)
questions.insert(0, "question_id", range(1, len(questions) + 1))
questions.to_csv(QUESTION_FILE, index=False)
print("Saved questions to", QUESTION_FILE)
questions[["question_id", "question", "reference_answer", "answerable"]]

Saved questions to /workspace/workshop_outputs/03_calibration_questions.csv


,question_id,question,reference_answer,answerable
0,1,What types of gastrointestinal cancers are ass...,MASLD is associated with an elevated risk of v...,1
1,2,What is the mean age of patients in the study?,The mean age of patients was 63.5 years.,1
2,3,What are the most common symptoms of postinfec...,Abdominal pain and diarrhea are the most commo...,1
3,4,What is the result of comparing the size of WJ...,WJSC-MT resulted to be larger than FIB-MT in a...,1
4,5,What is the effect of PTEN loss on glioma cell...,Loss of the PTEN tumor suppressor gene leads t...,1
5,6,What percentage of glioblastoma patients showe...,38% showed EGFR gene amplification.,1
6,7,What is the minimum viral load required for ve...,Viremia > 104 copies/ml can still infect vecto...,1
7,8,What is the outcome of the precancerous cell's...,The precancerous cell develops uncontrolled pr...,1
8,9,What drives the recurrence of glioblastoma mul...,"A shift to a mesenchymal phenotype, mediated b...",1
9,10,What percentage of participants had a diagnosi...,Forty percent (n=8) of participants had a diag...,1


## 4. ✋ Human review (do not skip)
Open the question CSV saved above (pilot or calibration) in a spreadsheet or the Jupyter CSV viewer:
- **delete** questions that are vague, trivial or wrong
- **edit** reference answers that are incomplete
- check that each question really is answerable from its `source_chunk`

Save the file, then run the next cell to reload it.

In [6]:
questions = pd.read_csv(QUESTION_FILE)
n_unanswerable = int((questions.answerable == 0).sum())
print(len(questions), "questions after review;", n_unanswerable, "unanswerable")
if CALIBRATION_MODE:
    if len(questions) < TARGET_QUESTIONS:
        raise ValueError(f"Keep at least {TARGET_QUESTIONS} reviewed questions; replace bad rows rather than deleting below target")
    if not 5 <= n_unanswerable <= 10:
        raise ValueError("Keep 5–10 unanswerable questions in the calibration set")

50 questions after review; 8 unanswerable


## 5. Run every question through MyGPT
This may take longer for the 50+ question calibration set.

In [7]:
answers = []
for i, r in questions.iterrows():
    print(f"[{i + 1}/{len(questions)}] {r.question}")
    try:
        res = client.ask(r.question)
    except Exception as exc:
        print("   failed:", exc)
        res = {"error": str(exc)}
    answers.append({**r.to_dict(), **{k: v for k, v in res.items() if k != "question"}})

answers = pd.DataFrame(answers)
answers.to_csv(ANSWER_FILE, index=False)
print("Saved answers to", ANSWER_FILE)
answers[["question_id", "question", "answer", "QRS", "ARS", "HI"]].head(10)

[1/50] What types of gastrointestinal cancers are associated with metabolic dysfunction-associated steatotic liver disease (MASLD)?
[2/50] What is the mean age of patients in the study?
[3/50] What are the most common symptoms of postinfectious irritable bowel syndrome (PI-IBS)?
[4/50] What is the result of comparing the size of WJSC-MT and FIB-MT over 28 days?
[5/50] What is the effect of PTEN loss on glioma cells' ability to escape the immune system?
[6/50] What percentage of glioblastoma patients showed EGFR gene amplification?
[7/50] What is the minimum viral load required for vector mosquitoes to be infected with dengue virus?
[8/50] What is the outcome of the precancerous cell's uncontrolled proliferation in the third stage?
[9/50] What drives the recurrence of glioblastoma multiforme (GBM)?
[10/50] What percentage of participants had a diagnosis of Generalized Anxiety Disorder?
[11/50] What percentage of children born outside the hospital of residence had the CMVc screening test

,question_id,question,answer,QRS,ARS,HI
0,1,What types of gastrointestinal cancers are ass...,"According to the provided context, metabolic d...",92.0,100.0,46.0
1,2,What is the mean age of patients in the study?,The answer is not explicitly stated in the pro...,82.0,93.0,52.0
2,3,What are the most common symptoms of postinfec...,"According to the provided context, the most co...",91.0,100.0,48.0
3,4,What is the result of comparing the size of WJ...,The answer to your question can be found in th...,82.0,92.0,53.0
4,5,What is the effect of PTEN loss on glioma cell...,"According to the provided context, the effect ...",90.0,100.0,52.0
5,6,What percentage of glioblastoma patients showe...,"According to the context, 38% of the 58 gliobl...",93.0,98.0,43.0
6,7,What is the minimum viral load required for ve...,"According to the provided context, the minimum...",88.0,99.0,56.0
7,8,What is the outcome of the precancerous cell's...,"According to the context, the outcome of the p...",87.0,100.0,51.0
8,9,What drives the recurrence of glioblastoma mul...,"According to the provided context, the recurre...",87.0,100.0,47.0
9,10,What percentage of participants had a diagnosi...,"According to the context, 40% (n=8) of partici...",80.0,88.0,52.0


➡️ Next: **03b_label_and_judge.ipynb**